In [11]:
"""
AI-Generated Email Evaluation System
Implements 6 evaluation metrics based on research framework
"""

# ============================================================================
# INSTALLATION & SETUP
# ============================================================================

# Install required packages
!pip install openai anthropic google-generativeai sentence-transformers scikit-learn pandas numpy tenacity -q

import os
import json
import time
import pandas as pd
import numpy as np
from typing import Dict, List, Tuple
from dataclasses import dataclass
from sklearn.cluster import KMeans
from sentence_transformers import SentenceTransformer
import openai
from anthropic import Anthropic
from tenacity import retry, stop_after_attempt, wait_random_exponential

# ============================================================================
# CONFIGURATION
# ============================================================================

class Config:
    """Configuration for API keys and model selection"""

    # ============================================================================
    # OPENROUTER CONFIGURATION
    # ============================================================================
    # Set your OpenRouter API key here
    OPENROUTER_API_KEY = "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"

    # OpenRouter base URL
    OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

    # Choose your model from OpenRouter's catalog
    # Examples:
    # - "openai/gpt-4o"
    # - "anthropic/claude-sonnet-4"
    # - "google/gemini-2.5-flash"
    # - "anthropic/claude-3.5-haiku"
    # - "openai/gpt-4o-mini"
    # - "google/gemma-3-4b-it"
    EVALUATION_MODEL = "openai/gpt-5-chat"


    # Semantic entropy settings
    N_SEMANTIC_SAMPLES = 5  # Number of outputs to generate for semantic entropy

    # API Pricing (USD per 1M tokens) - OpenRouter pricing
    PRICING = {
    # OpenAI Models
    'openai/gpt-5': {'input': 1.25, 'output': 10.0},
    'openai/gpt-5-chat': {'input': 1.25, 'output': 10.0},
    'openai/gpt-4o': {'input': 2.5, 'output': 10.0},
    'openai/gpt-4-turbo': {'input': 10.0, 'output': 30.0},
    'openai/gpt-4o-mini': {'input': 0.15, 'output': 0.6},

    # Anthropic Models
    'anthropic/claude-sonnet-4': {'input': 3.0, 'output': 15.0},
    'anthropic/claude-3.5-haiku': {'input': 0.8, 'output': 4.0},
    'anthropic/claude-3-haiku': {'input': 0.25, 'output': 1.25},

    # Google Models
    'google/gemini-2.5-flash': {'input': 0.3, 'output': 2.5},
    'google/gemini-2.5-pro': {'input': 1.25, 'output': 10.0},
    'google/gemma-3-4b-it': {'input': 0.017, 'output': 0.068},

    # Qwen Models
    'qwen/qwen-2.5-72b-instruct': {'input': 0.35, 'output': 0.4},
    'qwen/qwen2.5-vl-72b-instruct': {'input': 0.0, 'output': 0.0},
    'qwen/qwen3-coder-30b-a3b-instruct': {'input': 0.06, 'output': 0.25},

    # Meta Models
    'meta-llama/llama-3.3-70b-instruct': {'input': 0.35, 'output': 0.4},

    # Mistral Models
    'mistralai/mistral-small-3.2-24b-instruct:free': {'input': 0.0, 'output': 0.0},

    # xAI Models
    'x-ai/grok-4-fast': {'input': 0.20, 'output': 0.50},

    # Amazon Models
    'amazon/nova-micro-1.0': {'input': 0.035, 'output': 0.14},

    # DeepSeek Models
    'deepseek/deepseek-v3-0324': {'input': 0.24, 'output': 0.84},
    'deepseek/deepseek-v3': {'input': 0.3, 'output': 0.85},

    # Llama Models:
    'sao10k/l3-lunaris-8b': {'input': 0.04, 'output': 0.05},
    }

    @classmethod
    def setup(cls):
        """Setup OpenRouter client"""
        # Configure OpenAI client to use OpenRouter
        client = openai.OpenAI(
            api_key=cls.OPENROUTER_API_KEY,
            base_url=cls.OPENROUTER_BASE_URL,
            default_headers={
                "HTTP-Referer": "https://github.com/yourusername/email-eval",  # Optional
                "X-Title": "Email Evaluation System",  # Optional
            }
        )
        return {
            'openai': client,  # This client works for all OpenRouter models
            'openrouter': client
        }

    @classmethod
    def get_model_cost(cls, model: str, input_tokens: int, output_tokens: int) -> float:
        """Calculate cost for a model call"""
        if model not in cls.PRICING:
            # If model not in pricing dict, return 0 (unknown cost)
            print(f"⚠️  Warning: No pricing info for model '{model}'. Cost tracking disabled.")
            return 0.0

        pricing = cls.PRICING[model]
        input_cost = (input_tokens / 1_000_000) * pricing['input']
        output_cost = (output_tokens / 1_000_000) * pricing['output']
        return input_cost + output_cost

# ============================================================================
# DATA STRUCTURES
# ============================================================================

@dataclass
class EvaluationResult:
    """Store evaluation results for a single email"""
    hallucination_score: int  # Binary: 0 or 1
    # semantic_entropy: float
    cta_quality: int  # 1-10
    language_quality: int  # 1-10
    personalization: int  # 1-10
    human_likeness: int  # 1-10
    instruction_adherence: int  # 1-10

    overall_score: float
    is_acceptable: bool
    detailed_feedback: Dict

    # Cost tracking
    total_cost: float
    input_tokens: int
    output_tokens: int
    api_calls: int

    def to_dict(self):
        return {
            'hallucination_score': self.hallucination_score,
            # 'semantic_entropy': self.semantic_entropy,
            'cta_quality': self.cta_quality,
            'language_quality': self.language_quality,
            'personalization': self.personalization,
            'human_likeness': self.human_likeness,
            'instruction_adherence': self.instruction_adherence,
            'overall_score': self.overall_score,
            'is_acceptable': self.is_acceptable,
            'total_cost_usd': self.total_cost,
            'input_tokens': self.input_tokens,
            'output_tokens': self.output_tokens,
            'api_calls': self.api_calls,
            'detailed_feedback': self.detailed_feedback
        }

# ============================================================================
# SEMANTIC ENTROPY CALCULATOR (Hallucination Detection)
# ============================================================================

# class SemanticEntropyCalculator:
#     """
#     Implements semantic entropy-based hallucination detection
#     Based on Farquhar et al. (2024)
#     """

#     def __init__(self):
#         self.encoder = SentenceTransformer('all-MiniLM-L6-v2')

#     def generate_multiple_outputs(self, prompt: str, n: int, client) -> List[str]:
#         """Generate multiple outputs for the same prompt"""
#         outputs = []
#         for _ in range(n):
#             response = client.chat.completions.create(
#                 model=Config.EVALUATION_MODEL,  # Use the configured model
#                 messages=[{"role": "user", "content": prompt}],
#                 temperature=0.8
#             )
#             outputs.append(response.choices[0].message.content)
#         return outputs

#     def cluster_by_meaning(self, texts: List[str]) -> List[int]:
#         """Cluster texts by semantic similarity"""
#         embeddings = self.encoder.encode(texts)
#         n_clusters = min(len(texts), 3)
#         kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10) # Add n_init for KMeans
#         return kmeans.fit_predict(embeddings)

#     def calculate_entropy(self, cluster_labels: List[int]) -> float:
#         """Calculate entropy from cluster distribution"""
#         unique, counts = np.unique(cluster_labels, return_counts=True)
#         probabilities = counts / len(cluster_labels)
#         entropy = -np.sum(probabilities * np.log2(probabilities + 1e-10))
#         return entropy

#     def detect_hallucination(self, prompt: str, n_samples: int, client) -> Tuple[float, int]:
#         """
#         Detect hallucination using semantic entropy
#         Returns: (entropy_score, binary_hallucination_flag)
#         """
#         outputs = self.generate_multiple_outputs(prompt, n_samples, client)
#         clusters = self.cluster_by_meaning(outputs)
#         entropy = self.calculate_entropy(clusters)

#         # Threshold: entropy > 0.8 indicates potential hallucination
#         hallucination_flag = 1 if entropy > 0.8 else 0

#         return entropy, hallucination_flag

# ============================================================================
# LLM-AS-JUDGE EVALUATOR
# ============================================================================

class LLMEvaluator:
    """
    Uses LLM-as-judge approach for evaluating email quality
    Works with OpenRouter API
    """

    def __init__(self, model: str = Config.EVALUATION_MODEL):
        self.model = model
        self.clients = Config.setup()
        self.client = self.clients['openrouter']  # Use OpenRouter client for all models
        self.total_input_tokens = 0
        self.total_output_tokens = 0
        self.total_api_calls = 0
        self.total_cost = 0.0

    @retry(wait=wait_random_exponential(multiplier=1, max=60), stop=stop_after_attempt(3))
    def _call_llm(self, prompt: str) -> Tuple[str, int, int]:
        """
        Call LLM via OpenRouter with retry mechanism
        Returns: (response, input_tokens, output_tokens)
        """
        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.3,
                response_format={ "type": "json_object" } # Request JSON object
            )

            input_tokens = response.usage.prompt_tokens
            output_tokens = response.usage.completion_tokens
            content = response.choices[0].message.content

            # Track usage
            self.total_input_tokens += input_tokens
            self.total_output_tokens += output_tokens
            self.total_api_calls += 1

            # Calculate cost
            cost = Config.get_model_cost(self.model, input_tokens, output_tokens)
            self.total_cost += cost

            return content, input_tokens, output_tokens

        except openai.APIError as e:
            print(f"❌ API Error calling OpenRouter API: {e}")
            # Re-raise the exception to trigger retry
            raise
        except Exception as e:
            print(f"❌ Error calling OpenRouter API: {e}")
            raise

    def reset_usage(self):
        """Reset usage counters"""
        self.total_input_tokens = 0
        self.total_output_tokens = 0
        self.total_api_calls = 0
        self.total_cost = 0.0

    def get_usage_stats(self) -> Dict:
        """Get current usage statistics"""
        return {
            'total_input_tokens': self.total_input_tokens,
            'total_output_tokens': self.total_output_tokens,
            'total_api_calls': self.total_api_calls,
            'total_cost_usd': round(self.total_cost, 4)
        }

    @staticmethod
    def safe_json_loads(text: str):
        """Robustly parse possibly wrapped JSON from model output."""
        import json, re

        if not text or not text.strip():
            raise ValueError("Empty LLM response")

        s = text.strip()

        # 去掉 ```json ... ``` 包裹
        if s.startswith("```"):
            s = s.strip("` \n")
            if s.lower().startswith("json"):
                s = s[4:].strip()

        # 尝试直接解析
        try:
            return json.loads(s)
        except json.JSONDecodeError:
            pass

        # 提取第一个 { 到最后一个 } 再试
        start, end = s.find("{"), s.rfind("}")
        if 0 <= start < end:
            snippet = s[start:end+1]
            try:
                return json.loads(snippet)
            except json.JSONDecodeError:
                pass

        # 替换单引号为双引号重试
        s2 = re.sub(r"(?<!\\)'", '"', s)
        return json.loads(s2)

    def evaluate_hallucination(self, email: str, user_prompt: str) -> Tuple[int, str]:
        """Evaluate Hallucination (Binary Score: 0 or 1)"""
        prompt = f"""Evaluate the hallucination of this email on a binary scale of 0 or 1.

Email:
{email}

{f"user_prompt: {user_prompt}" if user_prompt else ""}

Description: A binary check for whether the model either fabricated information.

Score 0: No Hallucination
All specific claims are verifiable in the provided JSON data or are appropriately general statements.
Requirements:
All recipient details (name, title, company, work history) match JSON exactly
All sender company information (capabilities, metrics, value proposition) is from JSON
Any specific numbers, statistics, dates, or events are explicitly provided in JSON
General industry statements do not make unverifiable specific claims
No prohibited information mentioned (founding year, employee count, work history >4 years)
Content aligns with stated problems, persona, and solution domain in JSON
Data attribution is correct (no mixing recipient and sender information)
Score 1: Hallucination Detected
The email contains ANY of the following:
Fabricated specific facts not in JSON (statistics, metrics, events, activities, organizational details)
Prohibited mentions even if factually correct (founding year, employee count, work history >4 years old)
Incorrect use of provided data (wrong title, wrong company, misattribution)
Made-up recipient background or career information not in work history
Invented recent activities or signals not listed in JSON
Specific claims about recipient's company unsupported by JSON
Contextual misalignment (problems, solutions, or industry not matching JSON input)
Sender company capabilities or metrics not stated in JSON

Respond *only* with a JSON object containing 'score' (0 or 1) and 'reasoning' (string):
"""

        response_text, _, _ = self._call_llm(prompt)
        try:
            result = self.safe_json_loads(response_text)
            return result.get('score', -1), result.get('reasoning', 'No reasoning found')
        except Exception as e:
            print(f"❌ JSON Decode Error: {e}")
            print(f"Raw LLM response: {response_text}")
            return -1, f"Error parsing LLM response: {response_text[:200]}..."

    def evaluate_cta_quality(self, email: str) -> Tuple[int, str]:
        """Evaluate Call-to-Action quality (1-10 scale)"""
        prompt = f"""Evaluate the Call-to-Action (CTA) quality of this email on a scale of 1-10.

Email:
{email}

Evaluation Criteria:
- Clarity: Is the desired action clear?
- Relevance: Does it align with the email's purpose?
- Effectiveness: Is it compelling and low-friction?
- Appropriateness: Does it fit the context?
| **Score** | **Description**                                                                                                                                                                    |
| --------- | ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| 10    | Perfect CTA: extremely clear and specific (e.g., “15-minute call tomorrow at 3PM”), perfectly aligned with email flow, compelling and effortless to act on, tone perfectly suited. |
| 9     | Excellent CTA: clear, precise, and fully relevant; slightly less polished or natural phrasing than 10.                                                                             |
| 8     | Strong CTA: clear and aligned but could be more concise or smoother; minor friction in tone or structure.                                                                          |
| 7     | Good CTA: clear but slightly generic; connects to content but not fully persuasive; tone mostly appropriate.                                                                       |
| 6     | Fair CTA: understandable but vague (“schedule a chat”); weak motivation or moderate friction.                                                                                      |
| 5     | Average CTA: functional but uninspired; relevance partially missing; low perceived recipient value.                                                                                |
| 4     | Below Average CTA: somewhat confusing or poorly placed; uses soft or awkward phrasing.                                                                                             |
| 3     | Poor CTA: misaligned with email intent; pushy or inappropriately phrased; multiple minor rule violations.                                                                          |
| 2     | Very Poor CTA: unclear, high-friction, or includes explicit rule violations (“spare time,” “request”).                                                                             |
| 1     | Missing or nonsensical CTA: no actionable request, inappropriate tone, or completely irrelevant.                                                                                   |


Respond *only* with a JSON object containing 'score' (integer 1-10) and 'reasoning' (string):
"""

        response_text, _, _ = self._call_llm(prompt)
        try:
          result = self.safe_json_loads(response_text)
          return result.get('score', -1), result.get('reasoning', 'No reasoning found')
        except Exception as e:
            print(f"❌ JSON Decode Error: {e}")
            print(f"Raw LLM response: {response_text}")
            return -1, f"Error parsing LLM response: {response_text[:200]}..."

    def evaluate_language_quality(self, email: str) -> Tuple[int, str]:
        """Evaluate language quality and coherence (1-10 scale)"""
        prompt = f"""Evaluate the language quality and structural coherence of this email on a scale of 1-10.

Email:
{email}

Evaluation Criteria:
- Grammar and spelling
- Vocabulary appropriateness
- Sentence structure variety
- Conciseness (no unnecessary repetition)
- Punctuation and formatting

| **Score** | **Description**                                                                                                                                                          |
| --------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------ |
| 10    | Perfect: flawless grammar, spelling, and punctuation; rich but natural vocabulary; excellent sentence variety; concise and elegant writing; formatting enhances clarity. |
| 9     | Excellent: one or two very minor issues; strong, precise vocabulary; excellent readability and flow; nearly flawless formatting.                                         |
| 8     | Very Good: mostly error-free with only small grammatical slips; vocabulary fits tone and audience; minor repetition or slightly long sentences.                          |
| 7     | Good: clear and readable; occasional grammar or word choice errors; structure mostly varied; minor formatting inconsistencies.                                           |
| 6     | Fair: a few noticeable grammar or spelling mistakes; somewhat repetitive phrasing; acceptable but unpolished tone; some awkward constructions.                           |
| 5     | Average: several errors or awkward phrases; some inappropriate vocabulary; repetitive or overly long sentences; formatting slightly messy.                               |
| 4     | Below Average: frequent grammatical errors; inconsistent tone or vocabulary; poor sentence rhythm; noticeable repetition.                                                |
| 3     | Poor: major grammar or spelling problems; confusing or unnatural word choices; repetitive and clunky phrasing; poor readability.                                         |
| 2     | Very Poor: pervasive grammatical and vocabulary errors; extremely awkward flow; heavy repetition; formatting disrupts comprehension.                                     |
| 1     | Unacceptable: unreadable due to constant errors; inappropriate or nonsensical vocabulary; no structure or formatting coherence.                                          |


Respond *only* with a JSON object containing 'score' (integer 1-10) and 'reasoning' (string):
"""

        response_text, _, _ = self._call_llm(prompt)
        try:
            result = self.safe_json_loads(response_text)
            return result.get('score', -1), result.get('reasoning', 'No reasoning found')
        except Exception as e:
            print(f"❌ JSON Decode Error: {e}")
            print(f"Raw LLM response: {response_text}")
            return -1, f"Error parsing LLM response: {response_text[:200]}..."


    def evaluate_personalization(self, email: str, recipient_data: Dict) -> Tuple[int, str]:
        """Evaluate personalization and personalization (1-10 scale)"""
        prompt = f"""Evaluate how well this email is tailored to the specific recipient and addresses their relevant needs.

Email:
{email}

Recipient Data:
{json.dumps(recipient_data, indent=2)}

| **Score** | **Description**                                                                                                                                                                            |
| --------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------ |
| 10    | Deep personalization with multiple, specific data points seamlessly integrated; message feels handcrafted; perfectly aligns with recipient’s challenges and delivers high, tangible value. |
| 9     | Excellent personalization using rich, relevant context; smooth integration of details; strong value proposition highly aligned with role; nearly unique to this recipient.                 |
| 8     | Strong personalization; uses multiple relevant details naturally; good contextual understanding though slightly formulaic in parts; clear and valuable message.                            |
| 7     | Good personalization; uses one or two specific details; generally relevant and valuable, but phrasing or framing feels somewhat generic.                                                   |
| 6     | Moderate personalization; some recipient context used but shallowly integrated; message somewhat generic though still relevant.                                                            |
| 5     | Average personalization; mentions recipient/company but lacks depth; value proposition is broad and could apply to many others.                                                            |
| 4     | Below average; minimal context use; mechanical or misplaced personalization; unclear problem-solution relevance.                                                                           |
| 3     | Poor personalization; irrelevant or generic message; disconnected from recipient’s needs or timing.                                                                                        |
| 2     | Very poor personalization; data used incorrectly or out of context; content reads like a template.                                                                                         |
| 1     | No personalization at all; fully generic or irrelevant message; no identifiable recipient awareness.                                                                                       |


Respond *only* with a JSON object containing 'score' (integer 1-10) and 'reasoning' (string):
"""

        response_text, _, _ = self._call_llm(prompt)
        try:
            result = self.safe_json_loads(response_text)
            return result.get('score', -1), result.get('reasoning', 'No reasoning found')
        except Exception as e:
            print(f"❌ JSON Decode Error: {e}")
            print(f"Raw LLM response: {response_text}")
            return -1, f"Error parsing LLM response: {response_text[:200]}..."


    def evaluate_human_likeness(self, email: str) -> Tuple[int, str]:
        """Evaluate how human-like the email sounds (1-10 scale)"""
        prompt = f"""Evaluate whether this email reads like it was written by a real person with natural thought patterns and authentic voice.

Email:
{email}


| **Score** | **Description**                                                                                                                                                                   |
| --------- | --------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| 10    | Perfectly human-like: natural rhythm and flow; authentic voice; balanced tone; occasional minor phrasing quirks that enhance realism; feels indistinguishable from a real person. |
| 9     | Excellent: mostly human-sounding with strong natural variation; very minor awkward moments; tone and phrasing highly appropriate and genuine.                                     |
| 8     | Very Good: natural and smooth; slight over-polish or formulaic phrasing at times; emotional tone and style still feel authentic overall.                                          |
| 7     | Good: generally natural but slightly repetitive or too formal; minor mechanical patterns; small gaps in emotional or contextual depth.                                            |
| 6     | Fair: some sentences feel robotic or overly neutral; limited stylistic variation; tone appropriate but lacks warmth or spontaneity.                                               |
| 5     | Average: half natural, half mechanical; noticeable repetition and uniform rhythm; polite but impersonal tone.                                                                     |
| 4     | Below Average: clearly AI-generated patterns (“It’s important to note…”); overuse of cautious or filler phrases; stiff flow.                                                      |
| 3     | Poor: repetitive structure; emotionless or fake enthusiasm; generic and context-insensitive language.                                                                             |
| 2     | Very Poor: robotic tone throughout; constant structural uniformity; artificial politeness or inconsistent phrasing.                                                               |
| 1     | Completely AI-like: no human qualities; rigid, impersonal, overly formal or incoherent tone; zero variation or authenticity.                                                      |


Respond *only* with a JSON object containing 'score' (integer 1-10) and 'reasoning' (string):
"""

        response_text, _, _ = self._call_llm(prompt)
        try:
            result = self.safe_json_loads(response_text)
            return result.get('score', -1), result.get('reasoning', 'No reasoning found')
        except Exception as e:
            print(f"❌ JSON Decode Error: {e}")
            print(f"Raw LLM response: {response_text}")
            return -1, f"Error parsing LLM response: {response_text[:200]}..."

    def evaluate_instruction_adherence(self, email: str, instructions: str, user_prompt: str) -> Tuple[int, str]:
        """Evaluate adherence to given instructions (1-10 scale)"""
        prompt = f"""Evaluate how well this email adheres to the given instructions.

Email:
{email}

Instructions:
{instructions}

User Prompt:
{user_prompt}

Completeness: Does the email include all required components (e.g., CTA, personalization, brand voice, formatting)?
Accuracy: Are the instructions followed correctly, without deviation or omission?
Tone & Style: Does the email maintain the required tone (e.g., professional, friendly, persuasive) and style as specified?
Alignment with Goals: Does the email achieve the intended purpose or campaign objective?
Avoidance of Unnecessary Content: Does the email avoid including irrelevant, extra, or contradictory information?


| **Score** | **Description**                                                                                                                                                                                |
| --------- | ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| 10    | Perfect adherence: follows every instruction precisely; captures tone and structure flawlessly; includes all required elements with no deviation; clearly aligned with task goals and context. |
| 9     | Excellent: nearly perfect compliance with only trivial omissions; tone and purpose fully appropriate; all main components included; no significant extra or missing content.                   |
| 8     | Very Good: follows instructions well with minor inaccuracies; tone or phrasing slightly inconsistent but acceptable; includes all major elements.                                              |
| 7     | Good: adheres to most instructions; small gaps in detail or tone; minor structural inconsistencies; overall goal still achieved.                                                               |
| 6     | Fair: some deviations or omissions (e.g., missing small formatting or CTA detail); tone partially aligned; moderate unnecessary additions.                                                     |
| 5     | Average: noticeable gaps in following instructions; tone or purpose inconsistently applied; missing 1–2 key components (e.g., personalization or CTA).                                         |
| 4     | Below Average: several instruction failures; tone misaligned; missing multiple elements; includes some irrelevant or contradictory content.                                                    |
| 3     | Poor: major instruction violations; tone inappropriate for purpose; key required components omitted or misused.                                                                                |
| 2     | Very Poor: minimal instruction adherence; completely off-tone; multiple missing or irrelevant parts; fails to meet task intent.                                                                |
| 1     | Non-compliant: disregards instructions entirely; tone, purpose, and content fully misaligned; missing nearly all required components.                                                          |


Respond *only* with a JSON object containing 'score' (integer 1-10) and 'reasoning': '<one line, no line breaks>':
"""

        response_text, _, _ = self._call_llm(prompt)
        try:
            result = self.safe_json_loads(response_text)
            return result.get('score', -1), result.get('reasoning', 'No reasoning found')
        except Exception as e:
            print(f"❌ JSON Decode Error: {e}")
            print(f"Raw LLM response: {response_text}")
            return -1, f"Error parsing LLM response: {response_text[:200]}..."


# ============================================================================
# MAIN EVALUATION PIPELINE
# ============================================================================

class EmailEvaluationPipeline:
    """
    Complete evaluation pipeline for AI-generated emails
    Works with OpenRouter API
    """

    def __init__(self, model: str = Config.EVALUATION_MODEL):
        # self.entropy_calc = SemanticEntropyCalculator()
        self.llm_eval = LLMEvaluator(model)
        self.clients = Config.setup()
        self.openrouter_client = self.clients['openrouter']

    def evaluate_email(
        self,
        email: str,
        instructions: str = "",
        user_prompt: str = "",
        recipient_data: Dict = None,
        context: str = "",
        check_hallucination: bool = True
    ) -> EvaluationResult:
        """
        Comprehensive email evaluation

        Args:
            email: The email text to evaluate
            instructions: Original instructions for email generation
            user_prompt: User instructions for email generation
            recipient_data: Dict with recipient information (role, company, etc.)
            context: Additional context for relevance evaluation
            check_hallucination: Whether to run semantic entropy check (slower)
        """

        start_time = time.time()
        print("Starting evaluation...")
        detailed_feedback = {}

        # Reset usage counters for this evaluation
        self.llm_eval.reset_usage()

        # 1. Hallucination Detection (if enabled)
        # if check_hallucination:
        #     print("Checking for hallucinations...")
        #     # Use a prompt that explicitly asks for email generation for consistency check
        #     hallucination_prompt = f"Generate an email draft based on the following email:\n\n{email}"
        #     try:
        #         entropy, hi_score = self.entropy_calc.detect_hallucination(
        #             hallucination_prompt,
        #             Config.N_SEMANTIC_SAMPLES,
        #             self.openrouter_client  # Use OpenRouter client
        #         )
        #         detailed_feedback['hallucination'] = {
        #             'semantic_entropy': entropy,
        #             'interpretation': 'High uncertainty detected' if hi_score == 1 else 'Low uncertainty'
        #         }
        #     except Exception as e:
        #         print(f"❌ Error during hallucination check: {e}")
        #         entropy, hi_score = -1.0, -1 # Indicate error
        #         detailed_feedback['hallucination'] = {
        #             'semantic_entropy': entropy,
        #             'interpretation': f'Error during check: {e}'
        #         }
        # else:
        #     entropy, hi_score = 0.0, 0
        #     detailed_feedback['hallucination'] = {'semantic_entropy': 0.0, 'interpretation': 'Hallucination check skipped.'}


        # 1. Hallucination Detection
        print("Evaluating Hallucination Detection...")
        hi_score, hi_reasoning = self.llm_eval.evaluate_hallucination(email, user_prompt)
        detailed_feedback['hallucination'] = hi_reasoning
        print(f"HI score: {hi_score} — reason: {hi_reasoning[:200]}")

        # 2. CTA Quality
        print("Evaluating CTA quality...")
        cta_score, cta_reasoning = self.llm_eval.evaluate_cta_quality(email)
        detailed_feedback['cta'] = cta_reasoning

        # 3. Language Quality
        print("Evaluating language quality...")
        lq_score, lq_reasoning = self.llm_eval.evaluate_language_quality(email)
        detailed_feedback['language_quality'] = lq_reasoning

        # 4. personalization
        print("Evaluating personalization...")
        if recipient_data:
            per_score, per_reasoning = self.llm_eval.evaluate_personalization(
                email, recipient_data
            )
        else:
            per_score, per_reasoning = 5, "No recipient data provided"
        detailed_feedback['personalization'] = per_reasoning

        # 5. Human-likeness
        print("Evaluating human-likeness...")
        hl_score, hl_reasoning = self.llm_eval.evaluate_human_likeness(email)
        detailed_feedback['human_likeness'] = hl_reasoning

        # 6. Instruction Adherence
        print("Evaluating instruction adherence...")
        if instructions:
            ia_score, ia_reasoning = self.llm_eval.evaluate_instruction_adherence(
                email, instructions, user_prompt
            )
        else:
            ia_score, ia_reasoning = 5, "No instructions provided"
        detailed_feedback['instruction_adherence'] = ia_reasoning

        # Get usage statistics
        usage_stats = self.llm_eval.get_usage_stats()

        # Calculate overall score (weighted average)
        # Exclude hallucination score from overall average as it's binary
        scores = [cta_score, lq_score, per_score, hl_score, ia_score]
        # Filter out potential error scores (-1) before calculating average
        valid_scores = [score for score in scores if score != -1]
        overall_score = np.mean(valid_scores) if valid_scores else 0


        # Check acceptability conditions
        # Unacceptable if: HI == 1 OR AVG(scores) <= 6 OR LQ <= 3
        # Also consider hallucination_score = -1 as potentially unacceptable
        is_acceptable = (
            (hi_score == 0 or hi_score == -1) and # Treat error as potentially acceptable if other scores are good
            overall_score >= 6 and
            lq_score >= 6
        )

        elapsed_time = time.time() - start_time  # ⬅️ Added: calculate elapsed time
        detailed_feedback['runtime_seconds'] = elapsed_time  # ⬅️ Added: store runtime in feedback


        print(f"\nEvaluation complete!")
        print(f"💰 Cost: ${usage_stats['total_cost_usd']:.4f}")
        print(f"📊 Tokens: {usage_stats['total_input_tokens']} in / {usage_stats['total_output_tokens']} out")
        print(f"🔄 API Calls: {usage_stats['total_api_calls']}")
        print(f"⏱️ Runtime: {elapsed_time:.2f} seconds")  # ⬅️ Added: print runtime

        return EvaluationResult(
            hallucination_score=hi_score,
            # semantic_entropy=entropy,
            cta_quality=cta_score,
            language_quality=lq_score,
            personalization=per_score,
            human_likeness=hl_score,
            instruction_adherence=ia_score,
            overall_score=overall_score,
            is_acceptable=is_acceptable,
            detailed_feedback=detailed_feedback,
            total_cost=usage_stats['total_cost_usd'],
            input_tokens=usage_stats['total_input_tokens'],
            output_tokens=usage_stats['total_output_tokens'],
            api_calls=usage_stats['total_api_calls']
        )

    def evaluate_batch(
        self,
        emails: List[Dict],
        check_hallucination: bool = False
    ) -> pd.DataFrame:
        """
        Evaluate multiple emails in batch

        Args:
            emails: List of dicts with keys: 'email', 'instructions', 'user_prompt', 'recipient_data', 'context'
            check_hallucination: Whether to check hallucination (much slower)

        Returns:
            DataFrame with all evaluation results
        """
        results = []
        total_cost = 0.0
        batch_start = time.time()  # ⬅️ Added: record batch start time


        for i, email_data in enumerate(emails):
            print(f"\n{'='*60}")
            print(f"Evaluating email {i+1}/{len(emails)}")
            print(f"{'='*60}")

            result = self.evaluate_email(
                email=email_data['email'],
                instructions=email_data.get('instructions', ''),
                user_prompt=email_data.get('user_prompt', ''),
                recipient_data=email_data.get('recipient_data'),
                context=email_data.get('context', ''),
                check_hallucination=check_hallucination
            )

            total_cost += result.total_cost

            # results.append({
            #     'email_id': i,
            #     **result.to_dict()
            # })

            results.append({
                'email_id': i,
                **result.to_dict(),
                'runtime_seconds': result.detailed_feedback.get('runtime_seconds', None)  # ⬅️ Added: save per-email runtime
            })


        # Print batch summary
        print(f"\n{'='*60}")
        print(f"BATCH EVALUATION SUMMARY")
        print(f"{'='*60}")
        print(f"Total emails evaluated: {len(emails)}")
        print(f"💰 Total cost: ${total_cost:.4f}")
        print(f"💰 Average cost per email: ${total_cost/len(emails):.4f}")
        print(f"💰 Cost for 10,000 emails: ${(total_cost/len(emails))*10000:.2f}")
        batch_elapsed = time.time() - batch_start  # ⬅️ Added: calculate total batch runtime
        print(f"🕒 Total batch runtime: {batch_elapsed:.2f} seconds")  # ⬅️ Added
        print(f"⏳ Average runtime per email: {batch_elapsed/len(emails):.2f} seconds")  # ⬅️ Added


        return pd.DataFrame(results)



The following section is the same as above only with shorter prompt.

DO NOT RUN THE ABOVE AND THIS ONE AT THE SAME TIME.

In [12]:
# ============================================================================
# DATA LOADING FUNCTIONS
# ============================================================================

def load_test_data_from_file(file_path: str) -> List[Dict]:
    """
    Load test data from an external file.

    Supported formats:
    - JSON (.json): a JSON array containing test data

    Args:
        file_path: Path to the data file

    Returns:
        List of test data dictionaries
    """
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"❌ File not found: {file_path}")

    file_ext = os.path.splitext(file_path)[1].lower()

    try:
        if file_ext == '.json':
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
        else:
            raise ValueError(f"❌ Unsupported file format: {file_ext}. Supported format: .json")

        # Validate data structure
        required_fields = ['email']
        for i, item in enumerate(data):
            for field in required_fields:
                if field not in item:
                    raise ValueError(f"❌ Missing required field '{field}' in item {i}")

        print(f"✅ Successfully loaded {len(data)} test samples")
        return data

    except Exception as e:
        print(f"❌ Failed to load test data: {e}")
        raise


# ============================================================================
# MAIN TEST FUNCTIONS
# ============================================================================

def test_spreadsheet_examples(file_path: str):
    """Run evaluation pipeline on external test data from a JSON file"""

    print("\n" + "="*80)
    print("🧪 TESTING WITH EXTERNAL TEST DATA")
    print("="*80)

    # Load test data
    print(f"📁 Loading test data from file: {file_path}")
    test_data = load_test_data_from_file(file_path)

    if not test_data:
        print("❌ No test data found")
        return None

    # Initialize the evaluation pipeline
    try:
        pipeline = EmailEvaluationPipeline(model=Config.EVALUATION_MODEL)
    except Exception as e:
        print(f"\n❌ Error: Unable to initialize evaluation pipeline - {e}")
        return None

    # Evaluate all examples
    results_list = []

    for example in test_data:
        print(f"\n{'='*80}")
        print(f"📧 EVALUATING EXAMPLE {example.get('id', 'Unknown')}")
        recipient_data = example.get('recipient_data', {})
        print(f"Recipient: {recipient_data.get('name', 'Unknown')} - {recipient_data.get('role', 'Unknown')}")
        print(f"{'='*80}")

        # Print the email being evaluated
        print("\n📧 EMAIL BEING EVALUATED:")
        print("-" * 80)
        print(example['email'])
        print("-" * 80)

        # Print the evaluation context
        print("\n📋 EVALUATION CONTEXT:")
        print(f"Recipient Data: {recipient_data}")
        print(f"Context: {example.get('context', 'No context')}")
        print(f"Instructions: {example.get('instructions', 'No instructions provided')}")
        print(f"User Prompt: {example.get('user_prompt', 'No user prompt provided')}")


        result = pipeline.evaluate_email(
            email=example['email'],
            instructions=example.get('instructions', ''),
            user_prompt=example.get('user_prompt', ''),
            recipient_data=recipient_data,
            context=example.get('context', ''),
            check_hallucination=False  # disable for faster testing
        )

        results_list.append({
            'example_id': example.get('id', 'Unknown'),
            # 'recipient': recipient_data.get('name', 'Unknown'), # Removed recipient
            'overall_score': result.overall_score,
            'acceptable': result.is_acceptable,
            'hallucination': result.hallucination_score,
            'cta_quality': result.cta_quality,
            'language_quality': result.language_quality,
            'personalization': result.personalization,
            'human_likeness': result.human_likeness,
            'instruction_adherence': result.instruction_adherence,
            'cost_usd': result.total_cost
        })

    # Summary
    print("\n\n" + "="*80)
    print("📊 TEST RESULTS SUMMARY")
    print("="*80)

    df = pd.DataFrame(results_list)
    print(df.to_string(index=False))

    total_cost = df['cost_usd'].sum()
    avg_score = df['overall_score'].mean()
    acceptable_count = df['acceptable'].sum()

    print(f"\n{'='*80}")
    print(f"💰 Total Cost: ${total_cost:.4f}")
    print(f"💰 Average Cost per Email: ${total_cost/len(test_data):.4f}")
    print(f"💰 Estimated Cost for 10,000 emails: ${(total_cost/len(test_data))*10000:.2f}")
    print(f"\n📈 Average Overall Score: {avg_score:.2f}/10")
    print(f"✅ Acceptable Emails: {acceptable_count}/{len(test_data)} ({acceptable_count/len(test_data)*100:.0f}%)")
    print(f"{'='*80}\n")

    return df


def analyze_specific_example(file_path: str, example_id: int = 0):
    """Run detailed evaluation on a specific example"""

    print("\n" + "="*80)
    print(f"🔍 DETAILED ANALYSIS: EXAMPLE {example_id}")
    print("="*80)

    # Load test data
    test_data = load_test_data_from_file(file_path)

    # Find the specific example
    example = None
    for item in test_data:
        if item.get('id') == example_id:
            example = item
            break

    if not example:
        print(f"❌ Example with ID {example_id} not found")
        return None

    print("\n📧 EMAIL BEING EVALUATED:")
    print("-" * 80)
    print(example['email'])
    print("-" * 80)

    print("\n📋 EVALUATION CONTEXT:")
    recipient_data = example.get('recipient_data', {})
    print(f"Recipient: {recipient_data.get('name', 'Unknown')}")
    print(f"Role: {recipient_data.get('role', 'Unknown')}")
    print(f"Company: {recipient_data.get('company', 'Unknown')}")
    print(f"Context: {example.get('context', 'No context')}")

    print("\n📝 INSTRUCTIONS GIVEN:")
    print(example.get('instructions', 'No instructions provided'))

    pipeline = EmailEvaluationPipeline(model=Config.EVALUATION_MODEL)

    result = pipeline.evaluate_email(
        email=example['email'],
        instructions=example.get('instructions', ''),
        user_prompt=example.get('user_prompt', ''),
        recipient_data=recipient_data,
        context=example.get('context', ''),
        check_hallucination=False
    )

    print("\n" + "="*80)
    print("📊 EVALUATION RESULTS")
    print("="*80)
    print(f"Overall Score: {result.overall_score:.1f}/10")
    print(f"Acceptable: {'✅ YES' if result.is_acceptable else '❌ NO'}")
    print(f"\nBreakdown:")
    print(f"  CTA Quality: {result.cta_quality}/10")
    print(f"  Language Quality: {result.language_quality}/10")
    print(f"  personalization: {result.personalization}/10")
    print(f"  Human-likeness: {result.human_likeness}/10")
    print(f"  Instruction Adherence: {result.instruction_adherence}/10")
    print(f"\n💰 Cost: ${result.total_cost:.4f}")

    print("\n" + "="*80)
    print("💬 DETAILED FEEDBACK")
    print("="*80)
    for criterion, feedback in result.detailed_feedback.items():
        print(f"\n{criterion.upper()}:")
        if isinstance(feedback, dict):
            for key, value in feedback.items():
                print(f"  {key}: {value}")
        else:
            print(f"  {feedback}")

    return result


# ============================================================================
# MAIN EXECUTION ENTRY POINT
# ============================================================================

if __name__ == "__main__":
    print("\n🚀 Starting Email Evaluation System...")

    # Hardcoded file path for Colab
    file_path = "/content/hl.json"

    # Choose the mode you want: batch or specific analysis
    # Set this manually for now
    RUN_BATCH_TEST = True   # Set to False if you want to analyze a specific example
    EXAMPLE_ID = 0          # Only used if RUN_BATCH_TEST is False

    if RUN_BATCH_TEST:
        print(f"\n🧪 Running batch test from: {file_path}")
        results_df = test_spreadsheet_examples(file_path=file_path)
    else:
        print(f"\n🔍 Running detailed analysis on example ID {EXAMPLE_ID}")
        analyze_specific_example(file_path=file_path, example_id=EXAMPLE_ID)


🚀 Starting Email Evaluation System...

🧪 Running batch test from: /content/hl.json

🧪 TESTING WITH EXTERNAL TEST DATA
📁 Loading test data from file: /content/hl.json
✅ Successfully loaded 1 test samples

📧 EVALUATING EXAMPLE 10
Recipient: Unknown - Unknown

📧 EMAIL BEING EVALUATED:
--------------------------------------------------------------------------------
Hey Timothy,  

I saw your team at Brinks Home is driving quality in complex systems.  
How are you managing the challenges of upgrading legacy applications while keeping processes consistent for QA?  

We help modernize outdated systems so change feels natural.  
For example, we worked with Cornerstone Building Brands to replace 4,000+ network devices and standardize operations across 50 sites.  

Would you be open to a quick call to explore how we could do something similar for you?  

Best,  
--------------------------------------------------------------------------------

📋 EVALUATION CONTEXT:
Recipient Data: {"Prospect's w